In [1]:
import pandas as pd
import numpy as np

In [2]:
order_data = pd.read_csv("../Data/Processed/orders_cleaned.csv")
order_items_data=pd.read_csv("../Data/Processed/order_items_cleaned.csv")
customer_data=pd.read_csv("../Data/Processed/customers_cleaned.csv")

In [3]:
# How many total orders are there?
Total_orders=order_data["order_id"].count()
print(f"Total orders are:-",Total_orders)

Total orders are:- 99441


In [4]:
missing_value_analysis=order_data.groupby("order_status")[
    ["order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"]
    ].apply(lambda x:x.isna().mean()*100)
missing_value_analysis.round(2)

# 0% mean no missing value, 100% mean missing value present
# The missing values in the order table are largely explained by the order lifecycle rather than by data-quality issues. Orders in earlier stages such 
# as created, approved, processing, and invoiced naturally have missing shipping and delivery timestamps because those events have not yet occurred. 
# Similarly, shipped orders can have a missing customer-delivery date because delivery is still pending. Canceled and unavailable orders frequently 
# have missing downstream timestamps because they do not complete the normal fulfillment process. In contrast, missing timestamps for orders marked as 
# delivered are unexpected and should be investigated as potential data-quality issues.

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0.00,100.0,100.00
canceled,22.56,88.0,99.04
created,100.00,100.0,100.00
delivered,0.01,0.0,0.01
invoiced,0.00,100.0,100.00
processing,0.00,100.0,100.00
shipped,0.00,0.0,100.00
unavailable,0.00,100.0,100.00


In [5]:
delivered_status=order_data[
    (order_data['order_status']=='delivered')&
    (
        order_data['order_approved_at'].isna()|
        order_data["order_delivered_carrier_date"].isna()|
        order_data["order_delivered_customer_date"].isna()
    )
    ]
delivered_status

# "If approval is missing, how can the order be shipped and delivered?" Because logically, the order should normally have been approved before 
# progressing to shipping and delivery."The approval timestamp is missing even though subsequent fulfillment(shipped and order deliivered) events 
# occurred, suggesting an incomplete or missing event timestamp in the source data."
# The order clearly progressed beyond/without any approval, because it has shipping and delivery events.

# The customer delivery timestamp exists, so the order definitely reached the customer.Therefore, the missing carrier date is not logically 
# consistent with the normal event sequence.You can reasonably flag this as a potential data-quality issue or missing event timestamp.

# The order is marked delivered, but the customer delivery timestamp is missing.Since delivered means the delivery event has occurred, this missing 
# timestamp is unexpected.So yes, these records should be treated as potential data-quality issues.


# For delivered orders, timestamp completeness is expected because the order has completed the fulfillment lifecycle. However, a small number of 
# delivered orders have missing approval, carrier-delivery, or customer-delivery timestamps despite having subsequent lifecycle events. These cases 
# are inconsistent with the expected order sequence and should be treated as potential data-quality issues rather than structurally missing values.

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_year,order_purchase_month,order_purchase_month_name,order_delivery_time,delivery_delay_days,delivery_status
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:00,2017-11-28 17:56:00,2017-11-30 18:12:00,NaN,2017-12-18,2017,11,November,NaN,NaN,Unknown
5323,e04abd8149ef81b95221e88f6ed9ab6a,2127dc6603ac33544953ef05ec155771,delivered,2017-02-18 14:40:00,NaN,2017-02-23 12:04:00,2017-03-01 13:25:00,2017-03-17,2017,2,February,10.0,-16.0,Early
16567,8a9adc69528e1001fc68dd0aaebbb54a,4c1ccc74e00993733742a3c786dc3c1f,delivered,2017-02-18 12:45:00,NaN,2017-02-23 09:01:00,2017-03-02 10:05:00,2017-03-21,2017,2,February,11.0,-19.0,Early
19031,7013bcfc1c97fe719a7b5e05e61c12db,2941af76d38100e0f8740a374f1a5dc3,delivered,2017-02-18 13:29:00,NaN,2017-02-22 16:25:00,2017-03-01 08:07:00,2017-03-17,2017,2,February,10.0,-16.0,Early
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:00,2018-06-20 07:19:00,2018-06-25 08:05:00,NaN,2018-07-16,2018,6,June,NaN,NaN,Unknown
22663,5cf925b116421afa85ee25e99b4c34fb,29c35fc91fc13fb5073c8f30505d860d,delivered,2017-02-18 16:48:00,NaN,2017-02-22 11:23:00,2017-03-09 07:28:00,2017-03-31,2017,2,February,18.0,-22.0,Early
23156,12a95a3c06dbaec84bcfb0e2da5d228a,1e101e0daffaddce8159d25a8e53f2b2,delivered,2017-02-17 13:05:00,NaN,2017-02-22 11:23:00,2017-03-02 11:09:00,2017-03-20,2017,2,February,12.0,-18.0,Early
26800,c1d4211b3dae76144deccd6c74144a88,684cb238dc5b5d6366244e0e0776b450,delivered,2017-01-19 12:48:00,NaN,2017-01-25 14:56:00,2017-01-30 18:16:00,2017-03-01,2017,1,January,11.0,-30.0,Early
38290,d69e5d356402adc8cf17e08b5033acfb,68d081753ad4fe22fc4d410a9eb1ca01,delivered,2017-02-19 01:28:00,NaN,2017-02-23 03:11:00,2017-03-02 03:41:00,2017-03-27,2017,2,February,11.0,-25.0,Early
39334,d77031d6a3c8a52f019764e68f211c69,0bf35cac6cc7327065da879e2d90fae8,delivered,2017-02-18 11:04:00,NaN,2017-02-23 07:23:00,2017-03-02 16:15:00,2017-03-22,2017,2,February,12.0,-20.0,Early


In [6]:
# What percentage of orders are successfully delivered?
Delivered_orders=order_data[
    (order_data["order_status"]=="delivered")&
    (order_data["order_delivered_customer_date"].notna())
    ].shape[0]

Total_orders=order_data["order_id"].count()

Delivered_order_percentage=Delivered_orders/Total_orders*100

print(f"Total orders are:-",Total_orders)
print(f"Delivered orders are:-",Delivered_orders)
print(Delivered_order_percentage)

# This tells us that most of the orders around 97% of orders are delivered to the customers
# 97.0% of all recorded orders reached the customer successfully, indicating a high overall fulfillment rate.

Total orders are:- 99441
Delivered orders are:- 96470
97.01229875001258


In [9]:
# How many orders are placed each month?
order_over_month=order_data.groupby("order_purchase_month")[
    ["order_id"]
    ].nunique()

order_over_month#.sort_values("order_id",ascending=False)

# Highest order placed in august month and in september minimum order is placed in all the three years

,order_id
order_purchase_month,
1,8069
2,8508
3,9893
4,9343
5,10573
6,9412
7,10318
8,10843
9,4305


Merge different tables

In [11]:
customer_order=customer_data.merge(
    order_data[["order_id","customer_id","order_status","order_purchase_month","order_purchase_year"]],
    on="customer_id",
    how="left"
)

order_revenue=customer_order.merge(
    order_items_data[["order_id","price","freight_value"]],
    on="order_id",
    how="left"
)

order_customer=order_data.merge(
    customer_data[["customer_id","customer_unique_id","customer_state","customer_city"]],
    on="customer_id",
    how="left"
)

order_customer

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_purchase_year,order_purchase_month,order_purchase_month_name,order_delivery_time,delivery_delay_days,delivery_status,customer_unique_id,customer_state,customer_city
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:00,2017-10-02 11:07:00,2017-10-04 19:55:00,2017-10-10 21:25:00,2017-10-18,2017,10,October,8.0,-8.0,Early,7c396fd4830fd04220f754e42b4e5bff,SP,sao paulo
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-08-07 15:27:00,2018-08-13,2018,7,July,13.0,-6.0,Early,af07308b275d755c9edb36a90c618231,BA,barreiras
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-09-04,2018,8,August,9.0,-18.0,Early,3a653a41f6f9fc3d2a113cf8398680e8,GO,vianopolis
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-12-02 00:28:00,2017-12-15,2017,11,November,13.0,-13.0,Early,7c142cf63193a1473d2e66489a9ae977,RN,sao goncalo do amarante
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,2018,2,February,2.0,-10.0,Early,72632f0f9dd73dfee390c9b22eb56dd6,SP,santo andre
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:00,2017-03-09 09:54:00,2017-03-10 11:18:00,2017-03-17 15:08:00,2017-03-28,2017,3,March,8.0,-11.0,Early,6359f309b166b0196dbf7ad2ac62bb5a,SP,sao jose dos campos
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:00,2018-02-06 13:10:00,2018-02-07 23:22:00,2018-02-28 17:37:00,2018-03-02,2018,2,February,22.0,-2.0,Early,da62f9e57a76d978d02ab5362c509660,SP,praia grande
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:00,2017-08-27 15:04:00,2017-08-28 20:52:00,2017-09-21 11:24:00,2017-09-27,2017,8,August,24.0,-6.0,Early,737520a9aad80b3fbbdad19b66b37b30,BA,nova vicosa
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:00,2018-01-08 21:36:00,2018-01-12 15:35:00,2018-01-25 23:32:00,2018-02-15,2018,1,January,17.0,-21.0,Early,5097a5312c8b157bb7be58ae360ef43c,RJ,japuiba


In [13]:
# What is the monthly order trend over time?
Month_trend_over_time=(order_customer.groupby("order_purchase_month")
    .agg(
    customer_count=("customer_unique_id","nunique"),
    order_count=("order_id","nunique")
    )
)
Month_trend_over_time

# Monthly order volume showed a strong overall increase throughout 2017, with some month-to-month fluctuations, and reached its highest point in 
# November 2017. After declining in December, order volume remained relatively high throughout the first eight months of 2018.

,customer_count,order_count
order_purchase_month,,
1,7925,8069
2,8321,8508
3,9751,9893
4,9252,9343
5,10430,10573
6,9302,9412
7,10171,10318
8,10699,10843
9,4230,4305


In [14]:
order_customer = order_data.merge(
    customer_data[["customer_id", "customer_unique_id", "customer_state"]],
    on="customer_id",
    how="left"
)

In [16]:
# Which customer states have the highest delivery delays?
highest_delay_order_customer=(order_customer.groupby("customer_state")
                              .agg(
                                  customer_count=("customer_id","count"),
                                  delay_delivery=("delivery_delay_days","mean")
                              )
                             )
print(highest_delay_order_customer.sort_values("delay_delivery",ascending=False))

# order_revenue
# AC state has highest delivery delay                                         

                customer_count  delay_delivery
customer_state                                
AL                         413       -8.707809
MA                         747       -9.571827
SE                         350      -10.020896
ES                        2033      -10.496241
BA                        3380      -10.794533
CE                        1336      -10.804535
MS                         715      -11.052782
SP                       41746      -11.076380
PI                         495      -11.306723
SC                        3637      -11.508317
RJ                       12852      -11.766858
DF                        2140      -12.048077
TO                         280      -12.131387
GO                        2020      -12.185488
MG                       11635      -13.240775
PB                         536      -13.261122
PE                        1652      -13.293785
PR                        5045      -13.314239
RN                         485      -13.649789
RS           

In [18]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

order_data[order_date_columns] = (
    order_data[order_date_columns]
    .apply(pd.to_datetime, format="mixed", errors="coerce")
)

In [19]:
# Which stage of the order lifecycle takes the longest?

order_data["Approval_Time"]=(order_data["order_approved_at"]-order_data["order_purchase_timestamp"]).dt.days
order_data["Carrier_Processing_Time"]=(order_data["order_delivered_carrier_date"]-order_data["order_approved_at"]).dt.days
order_data["Delivery_Time"]=(order_data["order_delivered_carrier_date"]-order_data["order_delivered_customer_date"]).dt.days

Approval_lifecycle=order_data["Approval_Time"].mean()
Carrier_lifecycle=order_data["Carrier_Processing_Time"].mean()
Delivery_lifecycle=order_data["Delivery_Time"].mean()

print(Approval_lifecycle)
print(Carrier_lifecycle)
print(Delivery_lifecycle)


# According to the order lifecycle analysis, the Carrier → Customer delivery stage takes the longest, averaging approximately 8.88 days. This indicate
# that the delivery stage is the main operational bottleneck. The business should closely monitor this stage to identify the factors causing delays, 
# such as carrier performance, transportation time, geographic distance, or logistics issues, and take appropriate measures to reduce delivery time.

0.2705552925534594
2.3021588628077505
-9.876579424721431


In [20]:
# Order Status Distribution How many orders are delivered, canceled, shipped, processing, etc.?
status_distribution=order_data.groupby("order_status")[
    ["order_id"]
    ].count()
status_distribution

# delivered status has the highest numbers of orders it indicates that most of the orders are deliverd to the customers

,order_id
order_status,
approved,2
canceled,625
created,5
delivered,96478
invoiced,314
processing,301
shipped,1107
unavailable,609


In [22]:
# Seasonality:- Are there seasonal patterns in order volume?
seasonality=order_data.groupby("order_purchase_month")[
    ["order_id"]
    ].count()
seasonality

# There is no seasonality in order volume as the order placed is not commpleted in all months and their values are also not consistent over the months
# one of the reasons is that there is no conpleted order data for 2016 and 2018 years so we cannot tell anything about the order volume seasonality

# Better conclusion:-The available data does not provide sufficient evidence to reliably establish seasonal patterns because the observation period is
# incomplete for 2016 and 2018. A year-over-year comparison using only common months would provide a more reliable assessment.

,order_id
order_purchase_month,
1,8069
2,8508
3,9893
4,9343
5,10573
6,9412
7,10318
8,10843
9,4305


In [23]:
# Year-over-Year Growth How has order volume changed year-over-year?
yoy_growth=order_data.groupby("order_purchase_year")[
    ["order_id"]
    ].count()
yoy_growth["YOY growth"]=yoy_growth["order_id"].pct_change()*100
yoy_growth["volume_change_over_year"]=yoy_growth["order_id"].diff()
yoy_growth

# As we can see that our order in 2016 is very less while in 2017 its rapidly increased and in 2018 there is again drop in it this is beacause we have
# not full month data for 2016 and 2018

# Recorded order volume increased substantially from 2017 to 2018, but direct year-over-year comparison is not reliable because 2016 and 2018 contain
# incomplete periods. A like-for-like comparison over the same months should be used to assess true YoY growth.

,order_id,YOY growth,volume_change_over_year
order_purchase_year,,,
2016,329,NaN,NaN
2017,45101,13608.510638,44772.0
2018,54011,19.755660,8910.0


In [25]:
# Cancellation Rate:- What percentage of orders are canceled, and has this changed over time?

Total_status=order_data["order_status"].count()

canceled_status=order_data[
    (order_data["order_status"]=="canceled")&
    (order_data["order_delivered_customer_date"].isna())
    ].shape[0] 

cancellation_rate=canceled_status/Total_status*100

print(cancellation_rate)

monthly_cancellation=(order_data.groupby("order_purchase_month")
                      .agg(
                          total_orders=("order_id","count"),
                          canceled_orders=("order_status",lambda x:(x=="canceled").sum())
                      )
                     )
monthly_cancellation["cancellation_rate"]=(monthly_cancellation["canceled_orders"]/monthly_cancellation["total_orders"]*100)
print(monthly_cancellation)

# october 2018 has the highest cancellation rate as total order placed is 4 and later all order are canceled also as there is no consistence change 
# in canceled rate as the canceled rate is heavily depend on the state canceled order

# Better interpretation:-The overall cancellation rate is low at approximately 0.62%. Monthly rates fluctuate considerably, but extreme rates in month
# with very few orders—such as October 2018—should not be treated as representative.

0.6224796613067044
                      total_orders  canceled_orders  cancellation_rate
order_purchase_month                                                  
1                             8069               37           0.458545
2                             8508               90           1.057828
3                             9893               59           0.596381
4                             9343               33           0.353206
5                            10573               53           0.501277
6                             9412               34           0.361241
7                            10318               69           0.668734
8                            10843              111           1.023702
9                             4305               37           0.859466
10                            4959               54           1.088929
11                            7544               37           0.490456
12                            5674               11       

In [26]:
# Average Delivery Time:- How long does an order take to reach the customer?
avg_delivery_time=order_data["Delivery_Time"].mean()
print(avg_delivery_time)

-9.876579424721431


In [28]:
# Late Delivery Rate:- What percentage of orders are delivered later than expected?
order_data
Total_orders=order_data["order_id"].count()
Late_orders=order_data[
    (order_data["delivery_status"]=="late")
    ].shape[0]
Late_orders_percent=Late_orders/Total_orders*100
print(Late_orders_percent)

# Approximately 7.9% of recorded orders were delivered after the estimated delivery date, indicating that late delivery is a meaningful operational 
# issue despite the high overall fulfillment rate.

0.0


In [29]:
order_revenue = order_revenue.merge(
    order_data[["customer_id", "order_delivery_time", "delivery_status"]],
    on="customer_id",
    how="left"
)

In [30]:
# State-Level Delivery Performance:- Which customer states experience the longest delivery times or highest delay rates?
order_data
state_level_dustribution=(order_revenue.groupby("customer_state")
                          .agg(
                              delivery_time=("order_delivery_time","mean"),
                              delay_orders=("delivery_status", lambda x: (x == "late").sum()),
                              total_orders=("order_id","count")
                          )
                         )
state_level_dustribution["delay_rate"]=(state_level_dustribution["delay_orders"]/state_level_dustribution["total_orders"]*100)
state_level_dustribution.sort_values(["delivery_time","delay_rate"],ascending=False)

# From this we can say that RR has highest delivery_time and AL State has highest delay rate so these state has to work on their delivery facilities
# and AC state has lowest delay rate because out of 92 orders only 3 orders are late and SP has lowest delivery time that is why they have good 
# customer base

,delivery_time,delay_orders,total_orders,delay_rate
customer_state,,,,
RR,27.826087,0,52,0.0
AP,27.753086,0,82,0.0
AM,25.963190,0,166,0.0
AL,23.992974,0,446,0.0
PA,23.301708,0,1085,0.0
MA,21.203750,0,831,0.0
SE,20.978667,0,390,0.0
CE,20.537167,0,1487,0.0
AC,20.329670,0,92,0.0
